# 06 — Agentic RAG (Router)

**Priority:** 🟢 Nice-to-have — routing-logic breadth. *If skipped, revisit when:* when a single retrieval strategy doesn't fit all query types.

```
╔═══════════════════════════════════════════════════════════════╗
║               6. AGENTIC RAG (ROUTER)                         ║
║                                                               ║
║  Query ──► AI Agent (Router)                                  ║
║                   │                                           ║
║            Decides how to retrieve:                          ║
║                   ├── vector_search(query, source?) ──► VDB  ║
║                   ├── direct_answer(answer)                   ║
║                   └── calculator(expression)                  ║
║                   │         │                                 ║
║                   │    Embedding Model ◄──────── Vector DB    ║
║                   │         │                                 ║
║  Response ◄── Generative Model ◄── Prompt ◄── Context        ║
╚═══════════════════════════════════════════════════════════════╝
```

## What is Agentic RAG (Router)?

In all previous notebooks, the retrieval strategy was fixed: always retrieve from the vector DB, always with k=5. But in the real world:

- Some questions don't need retrieval at all ("What is 2+2?")
- Some questions need retrieval from a *specific* source ("Search only specs")
- Some questions need query reformulation ("rephrase this technical question more clearly")
- Some questions need a tool call ("calculate the cost")

**Agentic RAG (Router)** hands this decision to an LLM agent. The agent receives the user query and available tools, then decides:
1. Which tool to call (or to answer directly)
2. How to formulate the tool arguments
3. Whether to iterate (call multiple tools)

## Architecture

- **Router agent**: an LLM with tool-use capabilities
- **Tools**: `vector_search(query, source_filter?)`, `direct_answer(text)`, `calculator(expr)`, `bm25_search(query)`
- **Claude path**: uses Anthropic's native tool-use API
- **Local path**: Ollama tool calling with JSON-decision fallback

## What you'll learn
- How LLM tool-calling / function-calling works
- Build an agentic loop (observe → decide → act → observe)
- See how the agent picks different strategies for different query types
- Understand the tradeoffs vs static retrieval

In [ ]:
import sys; sys.path.insert(0, '..')
import ragkit.config as cfg

cfg.BACKEND = "claude"   # "claude" | "local"
print(f"Backend: {cfg.BACKEND}  |  Device: {cfg.DEVICE}")

## Step 1 — Build source-specific indexes

The router can direct searches to specific document categories (e.g., specs only, incidents only). We build one unified collection but use Chroma's metadata filtering.

In [ ]:
from ragkit.data import build_chunked_corpus
from ragkit.vectorstore import build_collection, query_collection, Hit
from rank_bm25 import BM25Okapi
import re

texts, metadatas = build_chunked_corpus(chunk_size=200, overlap=40)
collection = build_collection("helios_agentic", texts, metadatas, persist_dir="../.chroma")

# BM25 for exact-match queries
def tokenise(text):
    return re.findall(r'[a-z0-9]+(?:[\-\.][a-z0-9]+)*', text.lower())

bm25 = BM25Okapi([tokenise(t) for t in texts])

categories = set(m['category'] for m in metadatas)
print(f"Collection ready: {collection.count()} chunks")
print(f"Available categories: {sorted(categories)}")

## Step 2 — Define the tools

Tools are defined in a schema that the LLM can understand. Both Claude and Ollama use a similar JSON-based tool definition format.

In [ ]:
ROUTER_TOOLS = [
    {
        "name": "vector_search",
        "description": "Search the Helios Robotics knowledge base using semantic similarity. Use for conceptual questions, troubleshooting, procedures, and general product information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query (can be rephrased for better retrieval)"},
                "source_filter": {
                    "type": "string",
                    "description": "Optional category filter: spec, incident, team, project, proc, faq",
                    "enum": ["spec", "incident", "team", "project", "proc", "faq"]
                },
                "top_k": {"type": "integer", "description": "Number of results to return (default 5)", "default": 5}
            },
            "required": ["query"]
        }
    },
    {
        "name": "bm25_search",
        "description": "Search using exact keyword matching. Use for part numbers (HR-XXX-YYY), firmware versions (FW-XXX-X.Y.Z), incident IDs (INC-YYYY-NNN), or SKU codes.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The exact string or code to search for"},
                "top_k": {"type": "integer", "description": "Number of results", "default": 5}
            },
            "required": ["query"]
        }
    },
    {
        "name": "calculator",
        "description": "Evaluate a mathematical expression. Use for unit conversions, payload calculations, or any arithmetic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "A safe mathematical expression (e.g., '12 * 9.81', '8 * 3600')"}
            },
            "required": ["expression"]
        }
    },
    {
        "name": "direct_answer",
        "description": "Answer the question directly without searching — use ONLY when the answer is clearly a matter of general knowledge that doesn't require Helios-specific context (e.g., unit conversions, common sense, or if you already have sufficient context from previous tool calls).",
        "input_schema": {
            "type": "object",
            "properties": {
                "answer": {"type": "string", "description": "The direct answer to give to the user"}
            },
            "required": ["answer"]
        }
    },
]

print(f"Defined {len(ROUTER_TOOLS)} tools:")
for t in ROUTER_TOOLS:
    required = t['input_schema'].get('required', [])
    print(f"  {t['name']:20s} — {t['description'][:70]}")

## Step 3 — Tool execution engine

In [ ]:
import ast
import operator as op

# Safe calculator (no eval)
def safe_calc(expr: str) -> str:
    allowed_ops = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
                   ast.Div: op.truediv, ast.Pow: op.pow, ast.USub: op.neg}
    def _eval(node):
        if isinstance(node, ast.Constant): return node.value
        if isinstance(node, ast.BinOp):    return allowed_ops[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):  return allowed_ops[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported: {node}")
    try:
        result = _eval(ast.parse(expr, mode='eval').body)
        return f"{expr} = {result}"
    except Exception as e:
        return f"Error: {e}"

def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Execute a tool and return its result as a string."""
    
    if tool_name == "vector_search":
        query = tool_input["query"]
        k = tool_input.get("top_k", 5)
        source_filter = tool_input.get("source_filter")
        
        hits = query_collection(collection, query, k=k * 3 if source_filter else k)
        
        if source_filter:
            hits = [h for h in hits if h.metadata.get("category") == source_filter][:k]
        else:
            hits = hits[:k]
        
        if not hits:
            return "No results found."
        return "\n\n".join(f"[{h.metadata['source']} | score={h.score:.3f}]\n{h.text}" for h in hits)
    
    elif tool_name == "bm25_search":
        query = tool_input["query"]
        k = tool_input.get("top_k", 5)
        query_tokens = tokenise(query)
        scores = bm25.get_scores(query_tokens)
        top_idx = scores.argsort()[::-1][:k]
        results = [(float(scores[i]), texts[i], metadatas[i]) for i in top_idx if scores[i] > 0]
        if not results:
            return "No exact matches found for the given code/ID."
        return "\n\n".join(f"[{m['source']} | bm25={s:.2f}]\n{t}" for s, t, m in results)
    
    elif tool_name == "calculator":
        return safe_calc(tool_input["expression"])
    
    elif tool_name == "direct_answer":
        return tool_input["answer"]
    
    else:
        return f"Unknown tool: {tool_name}"

# Test each tool
print("Tool: calculator")
print("  ", execute_tool("calculator", {"expression": "12 * 9.81"}))
print()
print("Tool: bm25_search")
result = execute_tool("bm25_search", {"query": "HR-REED-UPGRADE", "top_k": 1})
print("  ", result[:150])

## Step 4 — The agentic loop

The agent loop:
1. LLM sees the user query + available tools
2. LLM decides on a tool call (or direct answer)
3. Tool is executed, result fed back to LLM
4. LLM either calls another tool or synthesises the final answer

This is the fundamental loop behind all LLM agents.

In [ ]:
from ragkit.llm import generate_tools, generate
from ragkit.pretty import show_agent_decision, show_answer
import json

ROUTER_SYSTEM = """You are a technical support router for Helios Robotics.

You have access to tools to answer user questions. For each question:
1. Decide which tool is most appropriate
2. Call that tool with the right arguments
3. Based on the tool result, either call another tool or synthesise the final answer

Guidelines:
- For part numbers (HR-XXX), firmware (FW-XXX), incident IDs (INC-YYYY-NNN): use bm25_search
- For conceptual/troubleshooting questions: use vector_search
- For questions about specific categories (only specs, only incidents): use source_filter
- For simple arithmetic: use calculator
- For general knowledge not specific to Helios: use direct_answer
- For complex questions: chain multiple tools

Always give a final synthesised answer to the user after collecting tool results."""

SYNTHESIS_SYSTEM = """You are a technical assistant for Helios Robotics.
Based on the tool results provided, synthesise a clear, precise answer.
Include part numbers, firmware versions, and incident IDs when relevant."""

def agentic_rag_router(question: str, max_steps: int = 3, verbose: bool = True) -> str:
    """
    Agentic RAG router: the LLM decides which tools to call and how.
    Returns the final synthesised answer.
    """
    tool_results = []
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Query: {question}")
        print(f"{'='*60}")
    
    for step in range(max_steps):
        # Build prompt with accumulated context
        context_so_far = ""
        if tool_results:
            context_so_far = "\n\nPrevious tool results:\n" + "\n---\n".join(
                f"[{r['tool']}({r['args']})]:\n{r['result'][:400]}" for r in tool_results
            )
        
        prompt = f"User question: {question}{context_so_far}"
        if tool_results:
            prompt += "\n\nBased on the tool results above, either call another tool if needed, or provide the final answer."
        
        # Ask the router agent
        decision = generate_tools(prompt, ROUTER_TOOLS, system=ROUTER_SYSTEM)
        
        if verbose:
            print(f"\nStep {step+1} — Agent decision:")
            show_agent_decision(decision)
        
        if not decision["tool_calls"]:
            # Agent answered directly (no tool call)
            final = decision["text"]
            break
        
        # Execute all tool calls
        for tc in decision["tool_calls"]:
            tool_name = tc["name"]
            tool_input = tc["input"]
            
            if tool_name == "direct_answer":
                final = tool_input.get("answer", "")
                if verbose:
                    print(f"  → Direct answer: {final[:150]}")
                return final
            
            result = execute_tool(tool_name, tool_input)
            tool_results.append({
                "tool": tool_name,
                "args": json.dumps(tool_input)[:80],
                "result": result
            })
            
            if verbose:
                print(f"  → Tool result ({len(result)} chars): {result[:200]}...")
    
    # Final synthesis
    all_context = "\n\n---\n\n".join(
        f"Tool: {r['tool']}({r['args']})\nResult:\n{r['result']}" for r in tool_results
    )
    final_prompt = f"Question: {question}\n\nTool results:\n{all_context}\n\nSynthesise a clear final answer:"
    final = generate(final_prompt, system=SYNTHESIS_SYSTEM)
    
    if verbose:
        show_answer(final, title="Final synthesised answer")
    
    return final

# Test 1: Should route to bm25_search (part number)
answer = agentic_rag_router("What does the part HR-REED-UPGRADE fix?")

In [ ]:
# Test 2: Should route to vector_search with source_filter=incident
answer = agentic_rag_router("Tell me about the incidents involving Joint 4")

In [ ]:
# Test 3: Should use calculator
answer = agentic_rag_router(
    "If the HeliosBase M1 runs at 1.0 m/s for 8 hours, how many km has it travelled?"
)

In [ ]:
# Test 4: Multi-tool (vector_search + synthesise)
answer = agentic_rag_router(
    "What are the key differences between the HeliosArm V2 and the upcoming V3, and who is leading the V3 project?"
)

## Step 5 — Observe routing decisions across query types

In [ ]:
import time

routing_tests = [
    ("part number",   "What gripper uses part number HR-EE-GRIP-01?"),
    ("incident ID",   "What happened in INC-2024-019?"),
    ("conceptual",    "Why does the Joint 4 problem only happen at high temperature?"),
    ("calculation",   "How many Wh does the HC-400 controller use in 8 hours of standby?"),
    ("category filter", "What specs are relevant to the HeliosBase M1 navigation?"),
]

print("Routing decisions by query type:")
print("=" * 70)

for typ, q in routing_tests:
    t0 = time.time()
    
    # Get only the first tool decision (don't execute)
    decision = generate_tools(
        f"User question: {q}",
        ROUTER_TOOLS,
        system=ROUTER_SYSTEM + "\n\nFor this test, just decide which tool to call. Don't synthesise an answer."
    )
    elapsed = time.time() - t0
    
    if decision["tool_calls"]:
        tc = decision["tool_calls"][0]
        args_str = json.dumps(tc["input"])[:60]
        print(f"  [{typ:16s}] → {tc['name']:20s} args={args_str}")
    else:
        print(f"  [{typ:16s}] → direct_answer")

print("\n→ The agent correctly routes exact codes to bm25_search")
print("→ and conceptual questions to vector_search")

## Step 6 — Latency profile

In [ ]:
import matplotlib.pyplot as plt
import time

test_q = "What is the payload of the HeliosArm V2?"

# Time each component
timings = {}

t0 = time.time()
decision = generate_tools(f"User question: {test_q}", ROUTER_TOOLS, system=ROUTER_SYSTEM)
timings['routing decision'] = time.time() - t0

if decision['tool_calls']:
    tc = decision['tool_calls'][0]
    t0 = time.time()
    result = execute_tool(tc['name'], tc['input'])
    timings['tool execution'] = time.time() - t0

t0 = time.time()
_ = generate(f"Question: {test_q}\nContext: {result}\nAnswer:", system=SYNTHESIS_SYSTEM)
timings['synthesis'] = time.time() - t0

# vs naive RAG
from ragkit.llm import generate as gen
t0 = time.time()
hits = query_collection(collection, test_q, k=5)
ctx = "\n\n".join(h.text for h in hits)
_ = gen(f"Context:\n{ctx}\n---\nQuestion: {test_q}")
timings['naive RAG total'] = time.time() - t0

fig, ax = plt.subplots(figsize=(8, 3))
labels = list(timings.keys())
vals = [timings[l] * 1000 for l in labels]
colors = ['#e74c3c', '#f39c12', '#27ae60', '#2b5797']
bars = ax.barh(labels, vals, color=colors[:len(labels)], alpha=0.85)
ax.bar_label(bars, fmt='%.0f ms', padding=3, fontsize=9)
ax.set_xlabel('Time (ms)')
ax.set_title('Agentic RAG latency breakdown')
plt.tight_layout(); plt.show()

total_agentic = sum(v for k, v in timings.items() if k != 'naive RAG total') * 1000
print(f"\nAgentic RAG total: {total_agentic:.0f} ms")
print(f"Naive RAG total:   {timings['naive RAG total']*1000:.0f} ms")
print(f"Overhead: {total_agentic/timings['naive RAG total']/10:.1f}× (routing decision is an extra LLM call)")

## Exercise — Routing a query to the right tool (languages, drag race, math)

An agentic router's first job is **classification**: read the query and pick the right tool *before* spending money on retrieval or generation. This exercise builds a small rule-based router over three tools and measures its accuracy on a labelled test set.

Tools: `language_kb` (grammar questions), `dragrace_kb` (Drag Race trivia), `calculator` (arithmetic).

1. **Implement**: Complete `route(query)` so each query goes to the correct tool. Arithmetic (digits + an operator) → `calculator`; drag keywords → `dragrace_kb`; grammar keywords → `language_kb`.
2. **Check**: Every query in `tests` should route correctly (accuracy = 100%).
3. **Reflect**: One sentence — what's a query that *both* keyword sets could match, and how would you break the tie?

In [ ]:
def route(query: str) -> str:
    q = query.lower()
    # arithmetic: contains a digit AND an operator
    if any(ch.isdigit() for ch in query) and any(op in query for op in "+-*/"):
        return "calculator"
    drag = ["queen", "lip sync", "runway", "drag race", "rupaul", "snatch game"]
    lang = ["subjunctive", "verb", "particle", "grammar", "conjugat", "tense", "pinyin", "hangul"]
    if any(w in q for w in drag):
        return "dragrace_kb"
    if any(w in q for w in lang):
        return "language_kb"
    return "language_kb"   # sensible default

# (query, expected tool)
tests = [
    ("Who won the Snatch Game challenge?",            "dragrace_kb"),
    ("How does the Spanish subjunctive work?",         "language_kb"),
    ("What is 12 * 3 + 5?",                             "calculator"),
    ("Explain the Korean topic particle grammar",      "language_kb"),
    ("Which queen had the best runway look?",          "dragrace_kb"),
]

correct = 0
for q, expected in tests:
    got = route(q)
    flag = "✅" if got == expected else "❌"
    correct += got == expected
    print(f"  {flag} {got:12s} (expected {expected:12s})  {q}")
accuracy = correct / len(tests)
print(f"\nRouting accuracy: {accuracy:.0%}")

# ── Task 3: an ambiguous query + tie-break idea (comment) ─────────────────────
#   Your answer:

# ── Self-check ────────────────────────────────────────────────────────────────
assert accuracy == 1.0, "every query should route to its correct tool"
print("\n✅ Exercise checks passed!")

## Tradeoffs

| Aspect | Naive RAG | Agentic Router |
|---|---|---|
| **Query routing** | Fixed strategy | Dynamic |
| **Tool choice** | Always vector search | Any tool (calculator, BM25, direct) |
| **Multi-step reasoning** | ★☆☆☆☆ | ★★★★☆ |
| **Latency** | ★★★★★ | ★★★☆☆ (+1 LLM call per step) |
| **Predictability** | ★★★★★ (deterministic) | ★★★☆☆ (routing can vary) |
| **Failure modes** | Wrong chunks retrieved | Wrong tool chosen; hallucinated tool args |
| **When to use** | Simple, uniform query types | Mixed queries needing different strategies |

## Exercises

1. **Add a tool**: Create a `search_person(name)` tool that searches specifically in the team documents. Does the router use it for people-related queries?
2. **Confuse the router**: Ask a question where you expect vector_search but the router picks bm25_search (or vice versa). Can you understand why?
3. **Multi-step chains**: Ask a question that requires both bm25_search AND vector_search. Does the agent call both? How does it synthesise?
4. **Local backend**: Switch `cfg.BACKEND = "local"`. How does the routing quality change with Ollama vs Claude?

**Next:** [07_agent_rag_multi_agent.ipynb](07_agent_rag_multi_agent.ipynb) — scale to multiple specialised agents.